# Faster R-CNN v2 (MobileNetV3-Large-FPN) on Pascal VOC 2012 — fast convergence, Colab

Second detector, compared against YOLO26. Same split, same subsets, same labels (difficult objects kept) as the YOLO notebook.

**What changed from v1, and why** (each change is measured in its own run):

| Change | Why |
|---|---|
| `min_size` 800 → **512** | VOC images are at most 500 px. 800 upscaled every image ~1.6–2.1×, i.e. ~2.4× more pixels than 512, for pixels that don't exist. |
| **COCO class rows copied** into the new 20-class predictor | All 20 VOC classes exist in COCO. Training starts from a detector that already works (like `cls_remap` in YOLO) instead of a random classifier. |
| **Freezing as an explicit list of modules** | Never the full model. The box head holds 72% of the parameters, so freezing only the backbone barely changes anything (see the table in *Model setup*). |
| **Linear warmup + per-iteration cosine** | v1 started at full LR from step 1 with no warmup; that hurts a freshly initialised head and especially the transferred COCO rows. |
| **Image cache** (decoded once to a uint8 file on local disk) | No JPEG decoding or XML parsing during training. |
| uint8 images to GPU, **colour jitter on GPU**, `pin_memory` | v1 converted to float32 and ran ColorJitter (hue = RGB→HSV per pixel) on 2 CPU cores. |
| Checkpoint to Google Drive every epoch, **resume** after a disconnect, best weights uploaded to W&B | Colab runtimes die; v1 saved nothing outside the session. |

**mAP is computed with torchmetrics (COCO-style, pycocotools backend), not with Ultralytics.** Compare YOLO and Faster R-CNN numbers with that caveat.

## Install
wandb, torchmetrics and pycocotools. PyTorch, torchvision and the kaggle CLI are preinstalled on Colab.

In [ ]:
!pip install -q -U wandb torchmetrics pycocotools

## Imports

In [ ]:
import math
import os
import random
import shutil
import time
import uuid
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import psutil
import torch
import wandb
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection import MeanAveragePrecision
from torchvision.io import ImageReadMode, read_image
from torchvision.models.detection import (FasterRCNN_MobileNet_V3_Large_FPN_Weights,
                                          fasterrcnn_mobilenet_v3_large_fpn)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import ColorJitter
from torchvision.utils import draw_bounding_boxes

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("device:", DEVICE, "|", GPU_NAME)
print("CPU cores:", os.cpu_count(), "| RAM:", round(psutil.virtual_memory().total / 1e9, 1), "GB",
      "| free disk on /content:", round(shutil.disk_usage("/content").free / 1e9, 1), "GB")

## Google Drive, secrets, W&B
Checkpoints go to Drive, because `/content` disappears when the runtime stops.

The secret names below are assumptions: rename them to whatever you called them in Colab's 🔑 Secrets panel
(Kaggle username + API key from your `kaggle.json`, and the W&B API key).

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
wandb.login(key=userdata.get("WANDB_API_KEY"))

## Data
Download the same Kaggle dataset as the YOLO notebook and unzip it to **local disk** (`/content`), not to Drive:
reading thousands of small files from Drive is very slow. Skipped if it's already there in this runtime.
The folders are located by searching, because the unzipped names can differ from the Kaggle mount.

In [ ]:
DATA_ROOT = Path("/content/voc")
if not DATA_ROOT.exists():
    !kaggle datasets download -d gopalbhattrai/pascal-voc-2012-dataset -p {DATA_ROOT} --unzip -q

TRAIN_DIR = next(DATA_ROOT.rglob("ImageSets/Main/trainval.txt")).parents[2]   # holds Annotations/, JPEGImages/, ImageSets/
TEST_DIR = next(p.parent for p in DATA_ROOT.rglob("Annotations") if "test" in str(p).lower())
print("train/val:", TRAIN_DIR, "\ntest:     ", TEST_DIR)

CKPT_ROOT = Path("/content/drive/MyDrive/voc_frcnn_v2")   # one folder per run, survives disconnects
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT = "voc-frcnn"   # same W&B project as v1; v2 runs are in group "v2-..."

NAMES = ("aeroplane,bicycle,bird,boat,bottle,bus,car,cat,chair,cow,diningtable,"
         "dog,horse,motorbike,person,pottedplant,sheep,sofa,train,tvmonitor").split(",")

## Split
Exactly the same procedure as in the YOLO notebook and v1: trainval.txt (11,540 detection images) → sorted → shuffled with seed 42 → first 80% train, rest val.
Test = every XML of the Kaggle test set.

In [ ]:
trainval_sorted = sorted((TRAIN_DIR / "ImageSets/Main/trainval.txt").read_text().split())
ids = list(trainval_sorted)
random.Random(42).shuffle(ids)
cut = int(len(ids) * 0.8)
train_ids, val_ids = ids[:cut], ids[cut:]
test_ids = sorted(p.stem for p in (TEST_DIR / "Annotations").glob("*.xml"))
print(len(train_ids), "train |", len(val_ids), "val |", len(test_ids), "test")   # expect 9232 | 2308 | 5138

### Fixed subsets
Same seeds, same train list, same order as the YOLO notebook, so these are the **same images YOLO used**.
- 32 images for the overfit sanity checks (train = val)
- 1,000 train images for the train-vs-val gap
- 10% of train: drawn only so the random state matches the YOLO notebook (the profiling cell replaces the timing run)

In [ ]:
random.seed(0)
overfit_ids = random.sample(train_ids, 32)

random.seed(1)
train_eval_ids = random.sample(train_ids, 1000)

random.seed(2)
timing_ids = random.sample(train_ids, len(train_ids) // 10)

## Config
Every run cell passes these explicitly, so changing a value here changes every run that uses it.

**Freezing presets.** Each preset is the list of modules whose weights don't update. Trainable parameter counts (measured on torchvision's model, 21-class predictor):

| Preset | Frozen | Trainable |
|---|---|---|
| `FREEZE_BACKBONE` | MobileNetV3 body | ~16.1 M of 19.0 M |
| `FREEZE_UP_TO_ROI` | body + FPN + RPN | ~14.0 M (the RoI box head + predictor) |
| `FREEZE_UP_TO_FC7` | body + FPN + RPN + `fc6` | ~1.2 M (`fc7` + predictor) |

Why the jump: the box head's first layer `fc6` maps the $7 \times 7 \times 256$ RoI feature to 1024 units,
$12544 \times 1024 \approx 12.8$ M weights, two-thirds of the whole model. The MobileNetV3 body is only ~3 M.

What makes freezing *faster*, not just smaller: autograd only builds a graph through modules that have something trainable below them.
With body + FPN + RPN frozen, the backward pass stops at the RoI features and never touches the feature maps.

In [ ]:
SEED = 0
EPOCHS = 6               # set after the profiling cell
BATCH = 8
NUM_WORKERS = os.cpu_count()   # usually 2 on free Colab; more workers than cores doesn't help

# optimizer (SGD, like torchvision's reference recipes; 0.01 for batch 8 = their 0.02 for batch 16)
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# schedule, counted in iterations (batches): linear warmup, then cosine down to LR * ETA_MIN_FACTOR
WARMUP_ITERS = 300           # ~1/4 of an epoch (9,232 / 8 = 1,154 iterations per epoch)
WARMUP_FACTOR = 0.001        # LR starts at LR * WARMUP_FACTOR
ETA_MIN_FACTOR = 0.01

# input size: shorter side -> MIN_SIZE, longer side capped at MAX_SIZE (same 800:1333 ratio as torchvision's default)
MIN_SIZE = 512
MAX_SIZE = 853

# augmentation: flip in the Dataset (it must move the boxes), colour jitter on the GPU inside train_model
HFLIP_PROB = 0.5
COLOR_JITTER = ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.02)

# freezing presets (module names inside torchvision's FasterRCNN)
FREEZE_BACKBONE = ["backbone.body"]
FREEZE_UP_TO_ROI = ["backbone.body", "backbone.fpn", "rpn"]
FREEZE_UP_TO_FC7 = ["backbone.body", "backbone.fpn", "rpn", "roi_heads.box_head.fc6"]

# COCO names that differ from VOC names (used to copy COCO's classifier rows)
VOC_TO_COCO = {"aeroplane": "airplane", "diningtable": "dining table", "motorbike": "motorcycle",
               "pottedplant": "potted plant", "sofa": "couch", "tvmonitor": "tv"}
WEIGHTS = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1

USE_AMP = True
EVAL_SCORE_THRESH = 0.001    # keep low-confidence boxes during evaluation (default 0.05 cuts the PR-curve tail)
SANITY_EPOCHS = 30

## Read the XML annotations
VOC boxes are pixel corners (`xmin, ymin, xmax, ymax`), counted from 1. torchvision wants 0-based corners, so `xmin - 1`, `ymin - 1`.
Class name → index in `NAMES` **+ 1**, because torchvision reserves 0 for background. Difficult objects are kept, like in YOLO.

In [ ]:
def parse_xml(xml_path):
    # VOC XML -> (boxes as [x1, y1, x2, y2] in pixels, labels 1-20)
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        name = obj.findtext("name").strip()
        bb = obj.find("bndbox")
        x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        x1, y1 = x1 - 1, y1 - 1
        if x2 > x1 and y2 > y1:                          # torchvision rejects zero-size boxes
            boxes.append([x1, y1, x2, y2])
            labels.append(NAMES.index(name) + 1)
    return boxes, labels

print("example", train_ids[0], "→", parse_xml(TRAIN_DIR / "Annotations" / f"{train_ids[0]}.xml"))

## Image cache (built once per runtime, ~1–3 min)
Every train/val image is decoded **once** and written as raw `uint8` pixels into one file on local disk (`/content/cache`).
The annotations are parsed in the same pass. After that, training never decodes a JPEG or parses an XML again:
an image is just a slice of that file, read through `np.memmap`.

- **Why uint8, not float32:** 4× smaller. 11,540 images × ~0.56 MB ≈ 6.5 GB on disk. The conversion to float happens on the GPU.
- **Why a file (memmap), not a Python list in RAM:** the OS page cache keeps the hot part in RAM and shares it between all
  DataLoader workers. A Python list of tensors would be copied into each worker over time, and free Colab has ~12 GB RAM.
- **The pass itself runs in the DataLoader workers** (decode + parse in parallel); the main process only appends bytes to the file.
- The index is saved **last**, so an interrupted build is never mistaken for a finished one.
- Only train/val is cached. Test is used once at the end and is read from the JPEGs.
- `/content` is wiped when the runtime stops, so a new runtime rebuilds the cache (same 1–3 min).

In [ ]:
class VOCDecode(Dataset):
    # Used only to build the cache: decode one JPEG to uint8 and parse its XML, inside a DataLoader worker.
    def __init__(self, ids, image_dir, ann_dir):
        self.ids, self.image_dir, self.ann_dir = ids, image_dir, ann_dir

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        image_id = self.ids[i]
        image = read_image(str(self.image_dir / f"{image_id}.jpg"), mode=ImageReadMode.RGB)   # uint8 [3, H, W]
        boxes, labels = parse_xml(self.ann_dir / f"{image_id}.xml")
        return image.permute(1, 2, 0).numpy(), boxes, labels                                  # [H, W, 3]


CACHE_DIR = Path("/content/cache")
CACHE_BIN = CACHE_DIR / "trainval_uint8.bin"
CACHE_INDEX = CACHE_DIR / "trainval_index.pt"

if not CACHE_INDEX.exists():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    offsets, heights, widths, all_boxes, all_labels = [], [], [], [], []
    position = 0
    decode_loader = DataLoader(VOCDecode(trainval_sorted, TRAIN_DIR / "JPEGImages", TRAIN_DIR / "Annotations"),
                               batch_size=32, shuffle=False, num_workers=NUM_WORKERS, collate_fn=list)
    with open(CACHE_BIN, "wb") as f:
        for batch in decode_loader:
            for image, boxes, labels in batch:
                h, w, _ = image.shape
                f.write(image.tobytes())                 # row-major H x W x 3 bytes
                offsets.append(position)
                heights.append(h)
                widths.append(w)
                all_boxes.append(boxes)
                all_labels.append(labels)
                position += h * w * 3
    torch.save({"ids": trainval_sorted, "offsets": offsets, "heights": heights, "widths": widths,
                "boxes": all_boxes, "labels": all_labels}, CACHE_INDEX)
    print(f"cache built in {(time.time() - t0) / 60:.1f} min, {position / 1e9:.2f} GB")

CACHE = torch.load(CACHE_INDEX)
print(len(CACHE["ids"]), "images in cache |", f"{CACHE_BIN.stat().st_size / 1e9:.2f} GB",
      "| largest image side:", max(max(CACHE["heights"]), max(CACHE["widths"])))

## Datasets
Both return `(image, target)`: the image as a **uint8** tensor `[3, H, W]` (0–255), and `{"boxes", "labels"}`.
The model input is made in `train_model` / `evaluate`: `.to(DEVICE)`, then `.float() / 255` on the GPU.

Horizontal flip (when `hflip_prob > 0`) happens here because the boxes have to flip with the image:
new $x_1 = W - x_2$, new $x_2 = W - x_1$. Colour jitter only changes pixels, so it runs on the GPU later.

In [ ]:
class VOCCached(Dataset):
    # Train/val images from the uint8 cache file, annotations from the cache index.
    def __init__(self, ids, cache, cache_bin, hflip_prob=0.0):
        self.ids = ids
        self.cache = cache
        self.cache_bin = cache_bin
        self.hflip_prob = hflip_prob
        self.position = {image_id: k for k, image_id in enumerate(cache["ids"])}   # image id -> row in the cache
        self.pixels = None   # opened lazily, so each DataLoader worker opens its own read-only memmap

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        if self.pixels is None:
            self.pixels = np.memmap(self.cache_bin, dtype=np.uint8, mode="r")
        k = self.position[self.ids[i]]
        start, H, W = self.cache["offsets"][k], self.cache["heights"][k], self.cache["widths"][k]
        image = torch.from_numpy(np.array(self.pixels[start:start + H * W * 3])).view(H, W, 3)
        image = image.permute(2, 0, 1).contiguous()                       # uint8 [3, H, W]

        boxes = torch.tensor(self.cache["boxes"][k], dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor(self.cache["labels"][k], dtype=torch.int64)

        if self.hflip_prob > 0 and torch.rand(1).item() < self.hflip_prob:
            image = image.flip(-1)                                        # mirror left-right
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        return image, {"boxes": boxes, "labels": labels}


class VOCFiles(Dataset):
    # Test images straight from JPEG + XML (used once at the end, so not worth caching).
    def __init__(self, ids, image_dir, ann_dir):
        self.ids, self.image_dir, self.ann_dir = ids, image_dir, ann_dir

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        image_id = self.ids[i]
        image = read_image(str(self.image_dir / f"{image_id}.jpg"), mode=ImageReadMode.RGB)   # uint8 [3, H, W]
        boxes, labels = parse_xml(self.ann_dir / f"{image_id}.xml")
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        return image, {"boxes": boxes, "labels": torch.tensor(labels, dtype=torch.int64)}


def collate(batch):
    # Images have different sizes, so a batch stays a tuple of lists instead of one stacked tensor.
    return tuple(zip(*batch))

### DataLoaders
`pin_memory=True` puts each batch in page-locked RAM so the copy to the GPU is faster and can overlap with compute (`non_blocking=True`).
`persistent_workers=True` keeps the workers alive between epochs instead of restarting them.

In [ ]:
def make_loader(dataset, shuffle):
    return DataLoader(dataset, batch_size=BATCH, shuffle=shuffle, num_workers=NUM_WORKERS, collate_fn=collate,
                      pin_memory=True, persistent_workers=NUM_WORKERS > 0)

train_loader_aug = make_loader(VOCCached(train_ids, CACHE, CACHE_BIN, hflip_prob=HFLIP_PROB), shuffle=True)
val_loader = make_loader(VOCCached(val_ids, CACHE, CACHE_BIN), shuffle=False)
train_eval_loader = make_loader(VOCCached(train_eval_ids, CACHE, CACHE_BIN), shuffle=False)
overfit_loader = make_loader(VOCCached(overfit_ids, CACHE, CACHE_BIN), shuffle=True)
overfit_eval_loader = make_loader(VOCCached(overfit_ids, CACHE, CACHE_BIN), shuffle=False)
test_loader = make_loader(VOCFiles(test_ids, TEST_DIR / "JPEGImages", TEST_DIR / "Annotations"), shuffle=False)
print(len(train_loader_aug), "train iterations per epoch")

### Data check
The same image twice: as stored, and forced through the flip (`hflip_prob=1.0`). The boxes must sit on the objects in both.

In [ ]:
check_id = val_ids[0]
plain, plain_t = VOCCached([check_id], CACHE, CACHE_BIN)[0]
flipped, flipped_t = VOCCached([check_id], CACHE, CACHE_BIN, hflip_prob=1.0)[0]
print("dtype", plain.dtype, "| shape", tuple(plain.shape), "| values", plain.min().item(), "-", plain.max().item())
print("labels:", [NAMES[l - 1] for l in plain_t["labels"].tolist()])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (img, t), title in zip(axes, [(plain, plain_t), (flipped, flipped_t)], ["cached", "flipped"]):
    drawn = draw_bounding_boxes(img, t["boxes"], [NAMES[l - 1] for l in t["labels"].tolist()], colors="red", width=3)
    ax.imshow(drawn.permute(1, 2, 0))
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Helper functions
Model setup, the training loop (like `train_model` in the FER notebook), evaluation and logging.

### Model setup: copying COCO's classifier rows
The COCO model's last layer has one row of weights per COCO class (91 rows, including background and unused "N/A" slots):
`cls_score` scores the class, `bbox_pred` has 4 rows per class for the box refinement.
A new 21-class predictor normally starts random, so the model first has to re-learn what a "dog" is.
Instead, for each VOC class we copy the matching COCO row (`dog` → `dog`, `sofa` → `couch`, …) and background → background.
The model then starts as a working VOC detector, and fine-tuning only adapts it. This is the Faster R-CNN version of YOLO's `cls_remap`.

In [ ]:
def build_model(frozen, coco_head, min_size, max_size):
    # COCO-pretrained Faster R-CNN with a 21-class predictor (20 VOC + background).
    #   frozen:    list of module names whose weights don't train (e.g. FREEZE_UP_TO_ROI)
    #   coco_head: True = initialise the predictor from the matching COCO rows, False = random
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=WEIGHTS, trainable_backbone_layers=6,   # freezing is done below
                                              min_size=min_size, max_size=max_size)
    coco_predictor = model.roi_heads.box_predictor            # 91 COCO classes
    predictor = FastRCNNPredictor(coco_predictor.cls_score.in_features, len(NAMES) + 1)

    if coco_head:
        coco_names = WEIGHTS.meta["categories"]                # index = COCO label id, 0 = "__background__"
        rows = [0] + [coco_names.index(VOC_TO_COCO.get(n, n)) for n in NAMES]
        box_rows = [4 * r + k for r in rows for k in range(4)] # bbox_pred has 4 outputs per class
        with torch.no_grad():
            predictor.cls_score.weight.copy_(coco_predictor.cls_score.weight[rows])
            predictor.cls_score.bias.copy_(coco_predictor.cls_score.bias[rows])
            predictor.bbox_pred.weight.copy_(coco_predictor.bbox_pred.weight[box_rows])
            predictor.bbox_pred.bias.copy_(coco_predictor.bbox_pred.bias[box_rows])
    model.roi_heads.box_predictor = predictor

    for name in frozen:
        model.get_submodule(name).requires_grad_(False)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    assert 0 < n_trainable < n_total, "every run must freeze part of the model"
    print(f"trainable parameters: {n_trainable / 1e6:.2f} M of {n_total / 1e6:.2f} M | frozen: {frozen}")
    return model.to(DEVICE), n_trainable, n_total

In [ ]:
@torch.inference_mode()
def evaluate(model, data_loader, class_metrics=False):
    # COCO-style mAP with torchmetrics. Returns (metrics dict, per-class W&B table or None).
    model.eval()
    model.roi_heads.score_thresh = EVAL_SCORE_THRESH
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=class_metrics)

    for images, targets in data_loader:
        images = [img.to(DEVICE, non_blocking=True).float() / 255 for img in images]   # uint8 -> float 0-1 on the GPU
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            preds = model(images)
        preds = [{"boxes": p["boxes"].float().cpu(), "scores": p["scores"].float().cpu(), "labels": p["labels"].cpu()}
                 for p in preds]
        metric.update(preds, list(targets))

    r = metric.compute()
    metrics = {"mAP50-95": r["map"].item(), "mAP50": r["map_50"].item(), "mAP75": r["map_75"].item(),
               "mAP_small": r["map_small"].item(), "mAP_medium": r["map_medium"].item(),
               "mAP_large": r["map_large"].item(), "mAR100": r["mar_100"].item()}
    table = None
    if class_metrics:
        rows = [[NAMES[c - 1], ap] for c, ap in zip(r["classes"].tolist(), r["map_per_class"].tolist())]
        table = wandb.Table(columns=["class", "AP50-95"], data=rows)
    return metrics, table

### Training loop with resume
Every epoch it saves `last.pt` (model, optimizer, scheduler, GradScaler, epoch, best mAP, history) to the run's Drive folder,
and `best.pt` (model weights only) when val mAP50-95 improves. If `last.pt` already exists, it continues from there,
so after a Colab disconnect you just re-run the same cells. **To restart a run from scratch, delete its Drive folder.**

The scheduler steps **once per batch**, because warmup and cosine are defined in iterations.

In [ ]:
def train_model(model, optimizer, scheduler, train_loader, val_loader, epochs, run_dir, color_jitter=None):
    # Train, validate every epoch, log to W&B, checkpoint to run_dir (resumable). Returns per-epoch history.
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    start_epoch, best_map, history = 1, -1.0, []

    if last_path.exists():
        ckpt = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch, best_map, history = ckpt["epoch"] + 1, ckpt["best_map"], ckpt["history"]
        print(f"resuming after epoch {ckpt['epoch']} (best val mAP50-95 so far {best_map:.3f})")

    for epoch in range(start_epoch, epochs + 1):
        # ---------- train ----------
        model.train()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        loss_sums, n_batches, n_images = {}, 0, 0
        for images, targets in train_loader:
            images = [img.to(DEVICE, non_blocking=True).float() / 255 for img in images]   # uint8 -> float 0-1 on the GPU
            if color_jitter is not None:
                images = [color_jitter(img) for img in images]                              # new random jitter per image
            targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]

            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                loss_dict = model(images, targets)          # 4 losses: RPN (2) + RoI head (2)
                loss = sum(loss_dict.values())
            if not math.isfinite(loss.item()):
                raise RuntimeError(f"loss is {loss.item()} at epoch {epoch}: lower the learning rate")

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()                             # once per batch

            for k, v in loss_dict.items():
                loss_sums[k] = loss_sums.get(k, 0.0) + v.item()
            loss_sums["loss_total"] = loss_sums.get("loss_total", 0.0) + loss.item()
            n_batches += 1
            n_images += len(images)

        train_sec = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]

        # ---------- validate ----------
        t1 = time.time()
        val, _ = evaluate(model, val_loader)
        val_sec = time.time() - t1

        log = {"epoch": epoch, "lr": lr, "time/train_sec": train_sec, "time/val_sec": val_sec,
               "time/train_imgs_per_sec": n_images / train_sec,
               "gpu/max_mem_gb": torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0,
               **{f"train/{k}": v / n_batches for k, v in loss_sums.items()},
               **{f"val/{k}": v for k, v in val.items()}}
        wandb.log(log)
        history.append(log)

        if val["mAP50-95"] > best_map:
            best_map = val["mAP50-95"]
            torch.save(model.state_dict(), best_path)
        tmp_path = run_dir / "last.tmp"                      # write, then rename: a disconnect mid-save can't corrupt last.pt
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict() if scheduler is not None else None,
                    "scaler": scaler.state_dict(), "epoch": epoch, "best_map": best_map, "history": history}, tmp_path)
        tmp_path.replace(last_path)

        print(f"epoch {epoch}/{epochs} | loss {log['train/loss_total']:.3f} | val mAP50-95 {val['mAP50-95']:.3f} "
              f"mAP50 {val['mAP50']:.3f} | train {train_sec / 60:.1f} min ({n_images / train_sec:.0f} img/s), "
              f"val {val_sec / 60:.1f} min")
    return history

### Logging helpers

In [ ]:
def wandb_run_id(run_dir):
    # The same W&B run ID for every attempt of this run, so a resumed Colab session continues the same W&B run.
    run_dir.mkdir(parents=True, exist_ok=True)
    id_file = run_dir / "wandb_id.txt"
    if not id_file.exists():
        id_file.write_text(uuid.uuid4().hex[:8])
    return id_file.read_text().strip()

In [ ]:
def log_fit_summary(history):
    # How training went: convergence speed (mAP after epoch 1, epoch reaching 95% of peak), peak, drop, timing.
    maps = [h["val/mAP50-95"] for h in history]
    best = max(range(len(maps)), key=lambda i: maps[i])
    reach95 = next(i for i, m in enumerate(maps) if m >= 0.95 * maps[best])
    summary = {
        "fit/epochs_run": len(history),
        "fit/val_mAP50-95_epoch1": maps[0],
        "fit/epoch_reaching_95pct_of_peak": reach95 + 1,
        "fit/best_epoch": best + 1,
        "fit/peak_val_mAP50-95": maps[best],
        "fit/last_val_mAP50-95": maps[-1],
        "fit/drop_from_peak": maps[best] - maps[-1],     # > 0: val got worse at the end -> overfitting hint
        "time/total_train_min": sum(h["time/train_sec"] for h in history) / 60,
        "time/mean_train_sec_per_epoch": sum(h["time/train_sec"] for h in history) / len(history),
        "time/mean_val_sec_per_epoch": sum(h["time/val_sec"] for h in history) / len(history),
        "time/mean_train_imgs_per_sec": sum(h["time/train_imgs_per_sec"] for h in history) / len(history),
    }
    wandb.run.summary.update(summary)
    print(f"epoch-1 val mAP50-95 {maps[0]:.3f} | 95% of peak at epoch {reach95 + 1} | "
          f"peak {maps[best]:.3f} (epoch {best + 1}) | drop from peak {summary['fit/drop_from_peak']:.3f}")

In [ ]:
def log_val_and_gap(model, best_path, val_loader, train_eval_loader):
    # Load the best weights, score them on val (with per-class AP) and on the 1,000 train images; log the gap.
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    val_m, val_table = evaluate(model, val_loader, class_metrics=True)
    train_m, _ = evaluate(model, train_eval_loader)
    gap = train_m["mAP50-95"] - val_m["mAP50-95"]

    wandb.run.summary.update({f"final/val_{k}": v for k, v in val_m.items()})
    wandb.run.summary.update({f"final/train_{k}": v for k, v in train_m.items()})
    wandb.run.summary["final/gap_mAP50-95"] = gap
    wandb.log({"per_class/val": val_table})
    print(f"val mAP50-95 {val_m['mAP50-95']:.3f} | train mAP50-95 {train_m['mAP50-95']:.3f} | gap {gap:+.3f}")

In [ ]:
def upload_checkpoint(best_path, artifact_name):
    # Upload the best weights to W&B as a model artifact (second copy besides Drive).
    artifact = wandb.Artifact(artifact_name, type="model")
    artifact.add_file(str(best_path))
    wandb.log_artifact(artifact)

## Profiling: where does the time go?
Before any long run: time the **DataLoader alone** and the **model step alone** (forward + backward + optimizer, one fixed batch on the GPU)
for each freezing preset. An epoch can't be faster than the slower of the two.
- loader slower → CPU-bound: more caching or less CPU work per image would help, a faster GPU wouldn't
- model step slower → GPU-bound: smaller input or more freezing helps, more workers wouldn't

No W&B run; the numbers are printed and go into the report.

In [ ]:
# 1) DataLoader alone (training loader with flip)
it = iter(train_loader_aug)
next(it)                                   # start the workers
t0 = time.time()
for _ in range(40):
    images, targets = next(it)
loader_sec = (time.time() - t0) / 40
print(f"DataLoader: {loader_sec * 1000:.0f} ms per batch of {BATCH}  ({BATCH / loader_sec:.0f} img/s)")

# 2) model step alone, same fixed batch, for each freezing preset
gpu_images = [img.to(DEVICE).float() / 255 for img in images]
gpu_targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
steps_per_epoch = len(train_loader_aug)
for preset_name, frozen in [("FREEZE_BACKBONE", FREEZE_BACKBONE), ("FREEZE_UP_TO_ROI", FREEZE_UP_TO_ROI),
                            ("FREEZE_UP_TO_FC7", FREEZE_UP_TO_FC7)]:
    torch.manual_seed(SEED)
    model, _, _ = build_model(frozen, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)
    model.train()
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR * WARMUP_FACTOR, momentum=MOMENTUM)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    for i in range(25):
        if i == 5:                             # the first steps include CUDA warm-up, don't time them
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            loss = sum(model(gpu_images, gpu_targets).values())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    step_sec = (time.time() - t0) / 20
    bottleneck = "GPU" if step_sec > loader_sec else "CPU / DataLoader"
    print(f"  {preset_name}: model step {step_sec * 1000:.0f} ms per batch -> bottleneck {bottleneck}, "
          f"estimated train epoch ≈ {steps_per_epoch * max(step_sec, loader_sec) / 60:.1f} min")
    del model, optimizer
torch.cuda.empty_cache()

## Sanity checks: overfit 32 images
One per freezing preset (every trainable-parameter change gets one). Train and validate on the same 32 images, no augmentation,
constant LR after the warmup, `SANITY_EPOCHS` epochs. PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

For A the predictor starts random (the hardest case); for B and C it starts from the COCO rows, as in their real runs.

### Sanity A — backbone frozen, random predictor

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_BACKBONE, coco_head=False, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(        # short warmup, then constant
    optimizer, lambda step: min(1.0, WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / 50))

RUN_DIR = CKPT_ROOT / "sanity_A"
wandb.init(project=PROJECT, name="v2 Sanity A | overfit 32 | backbone frozen, random predictor", group="v2-sanity",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "purpose": "overfit 32 images", "frozen": FREEZE_BACKBONE, "coco_head": False,
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "warmup_iters": 50, "epochs": SANITY_EPOCHS,
    "min_size": MIN_SIZE, "max_size": MAX_SIZE, "augmentation": "none", "gpu": GPU_NAME,
})
history = train_model(model, optimizer, scheduler, overfit_loader, overfit_eval_loader, SANITY_EPOCHS, RUN_DIR)

peak = max(h["val/mAP50"] for h in history)
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

### Sanity B — body + FPN + RPN frozen, COCO rows

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_UP_TO_ROI, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(        # short warmup, then constant
    optimizer, lambda step: min(1.0, WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / 50))

RUN_DIR = CKPT_ROOT / "sanity_B"
wandb.init(project=PROJECT, name="v2 Sanity B | overfit 32 | body + FPN + RPN frozen, COCO rows", group="v2-sanity",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "purpose": "overfit 32 images", "frozen": FREEZE_UP_TO_ROI, "coco_head": True,
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "warmup_iters": 50, "epochs": SANITY_EPOCHS,
    "min_size": MIN_SIZE, "max_size": MAX_SIZE, "augmentation": "none", "gpu": GPU_NAME,
})
history = train_model(model, optimizer, scheduler, overfit_loader, overfit_eval_loader, SANITY_EPOCHS, RUN_DIR)

peak = max(h["val/mAP50"] for h in history)
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

### Sanity C — body + FPN + RPN + fc6 frozen, COCO rows

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_UP_TO_FC7, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(        # short warmup, then constant
    optimizer, lambda step: min(1.0, WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / 50))

RUN_DIR = CKPT_ROOT / "sanity_C"
wandb.init(project=PROJECT, name="v2 Sanity C | overfit 32 | body + FPN + RPN + fc6 frozen, COCO rows", group="v2-sanity",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "purpose": "overfit 32 images", "frozen": FREEZE_UP_TO_FC7, "coco_head": True,
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "warmup_iters": 50, "epochs": SANITY_EPOCHS,
    "min_size": MIN_SIZE, "max_size": MAX_SIZE, "augmentation": "none", "gpu": GPU_NAME,
})
history = train_model(model, optimizer, scheduler, overfit_loader, overfit_eval_loader, SANITY_EPOCHS, RUN_DIR)

peak = max(h["val/mAP50"] for h in history)
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## R0 — zero-shot: COCO rows copied, no training
How good is the model **before** any VOC training? This is the head start that the COCO-row copy gives (compare with YOLO's R0).
Evaluation only, ~1 min.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_UP_TO_ROI, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)

RUN_DIR = CKPT_ROOT / "R0_zero_shot"
wandb.init(project=PROJECT, name="v2 R0 | zero-shot, COCO rows, no training", group="v2-ablation",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "run": "R0", "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "COCO rows copied for the 20 VOC classes + background", "training": "none",
    "min_size": MIN_SIZE, "max_size": MAX_SIZE, "eval_score_thresh": EVAL_SCORE_THRESH,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)", "gpu": GPU_NAME,
})
val_m, val_table = evaluate(model, val_loader, class_metrics=True)
wandb.run.summary.update({f"final/val_{k}": v for k, v in val_m.items()})
wandb.log({"per_class/val": val_table})
print(val_m)
wandb.finish()

## R1 — v2 baseline: backbone frozen, random predictor
The v1 recipe with the v2 speed changes (512 px, cache, GPU jitter, warmup + cosine) and the backbone frozen.
The predictor starts **random**, like v1. Reference point for R2.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_BACKBONE, coco_head=False, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
total_iters = EPOCHS * len(train_loader_aug)
def lr_lambda(step):                                    # multiplier on LR at this iteration
    if step < WARMUP_ITERS:                             # linear warmup: LR * WARMUP_FACTOR -> LR
        return WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / WARMUP_ITERS
    progress = (step - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)   # 0 -> 1 over the rest of the run
    return ETA_MIN_FACTOR + (1 - ETA_MIN_FACTOR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

RUN_DIR = CKPT_ROOT / "R1_backbone_frozen_random_head"
wandb.init(project=PROJECT, name="v2 R1 | backbone frozen, random predictor", group="v2-ablation",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "run": "R1",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": 'new 21-class predictor (random init)',
    "coco_head": False, "frozen": FREEZE_BACKBONE, "frozen_desc": "MobileNetV3 body",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "linear warmup + cosine, per iteration", "warmup_iters": WARMUP_ITERS,
    "warmup_factor": WARMUP_FACTOR, "eta_min_factor": ETA_MIN_FACTOR,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter on GPU", "hflip_prob": HFLIP_PROB,
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "image_cache": "uint8 memmap", "num_workers": NUM_WORKERS, "pin_memory": True,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED, "gpu": GPU_NAME,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, RUN_DIR, color_jitter=COLOR_JITTER)

log_fit_summary(history)
log_val_and_gap(model, RUN_DIR / "best.pt", val_loader, train_eval_loader)
upload_checkpoint(RUN_DIR / "best.pt", "frcnn-v2-R1")
wandb.finish()

## R2 — R1 + COCO rows copied into the predictor
**Only change vs R1:** the predictor starts from COCO's rows. Compare `fit/val_mAP50-95_epoch1` and `fit/epoch_reaching_95pct_of_peak`:
that is the convergence effect of the transfer.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_BACKBONE, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
total_iters = EPOCHS * len(train_loader_aug)
def lr_lambda(step):                                    # multiplier on LR at this iteration
    if step < WARMUP_ITERS:                             # linear warmup: LR * WARMUP_FACTOR -> LR
        return WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / WARMUP_ITERS
    progress = (step - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)   # 0 -> 1 over the rest of the run
    return ETA_MIN_FACTOR + (1 - ETA_MIN_FACTOR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

RUN_DIR = CKPT_ROOT / "R2_backbone_frozen_coco_head"
wandb.init(project=PROJECT, name="v2 R2 | backbone frozen, COCO rows", group="v2-ablation",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "run": "R2",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": 'COCO rows copied for the 20 VOC classes + background',
    "coco_head": True, "frozen": FREEZE_BACKBONE, "frozen_desc": "MobileNetV3 body",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "linear warmup + cosine, per iteration", "warmup_iters": WARMUP_ITERS,
    "warmup_factor": WARMUP_FACTOR, "eta_min_factor": ETA_MIN_FACTOR,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter on GPU", "hflip_prob": HFLIP_PROB,
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "image_cache": "uint8 memmap", "num_workers": NUM_WORKERS, "pin_memory": True,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED, "gpu": GPU_NAME,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, RUN_DIR, color_jitter=COLOR_JITTER)

log_fit_summary(history)
log_val_and_gap(model, RUN_DIR / "best.pt", val_loader, train_eval_loader)
upload_checkpoint(RUN_DIR / "best.pt", "frcnn-v2-R2")
wandb.finish()

## R3 — R2 with body + FPN + RPN frozen (only the RoI head trains)
**Only change vs R2:** FPN and RPN are frozen too. The RPN's proposals are class-agnostic ("is there an object here"),
which COCO already does well for VOC's 20 classes. Backward stops at the RoI features, so expect faster epochs; the question is the accuracy cost.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_UP_TO_ROI, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
total_iters = EPOCHS * len(train_loader_aug)
def lr_lambda(step):                                    # multiplier on LR at this iteration
    if step < WARMUP_ITERS:                             # linear warmup: LR * WARMUP_FACTOR -> LR
        return WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / WARMUP_ITERS
    progress = (step - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)   # 0 -> 1 over the rest of the run
    return ETA_MIN_FACTOR + (1 - ETA_MIN_FACTOR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

RUN_DIR = CKPT_ROOT / "R3_up_to_roi_frozen_coco_head"
wandb.init(project=PROJECT, name="v2 R3 | body+FPN+RPN frozen, COCO rows", group="v2-ablation",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "run": "R3",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": 'COCO rows copied for the 20 VOC classes + background',
    "coco_head": True, "frozen": FREEZE_UP_TO_ROI, "frozen_desc": "body + FPN + RPN",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "linear warmup + cosine, per iteration", "warmup_iters": WARMUP_ITERS,
    "warmup_factor": WARMUP_FACTOR, "eta_min_factor": ETA_MIN_FACTOR,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter on GPU", "hflip_prob": HFLIP_PROB,
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "image_cache": "uint8 memmap", "num_workers": NUM_WORKERS, "pin_memory": True,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED, "gpu": GPU_NAME,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, RUN_DIR, color_jitter=COLOR_JITTER)

log_fit_summary(history)
log_val_and_gap(model, RUN_DIR / "best.pt", val_loader, train_eval_loader)
upload_checkpoint(RUN_DIR / "best.pt", "frcnn-v2-R3")
wandb.finish()

## R4 — R3 with `fc6` frozen too (only `fc7` + predictor train, ~1.2 M parameters)
**Only change vs R3:** the big `fc6` layer (12.8 M weights) is frozen. The most aggressive freezing level.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(FREEZE_UP_TO_FC7, coco_head=True, min_size=MIN_SIZE, max_size=MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
total_iters = EPOCHS * len(train_loader_aug)
def lr_lambda(step):                                    # multiplier on LR at this iteration
    if step < WARMUP_ITERS:                             # linear warmup: LR * WARMUP_FACTOR -> LR
        return WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / WARMUP_ITERS
    progress = (step - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)   # 0 -> 1 over the rest of the run
    return ETA_MIN_FACTOR + (1 - ETA_MIN_FACTOR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

RUN_DIR = CKPT_ROOT / "R4_up_to_fc7_frozen_coco_head"
wandb.init(project=PROJECT, name="v2 R4 | everything but fc7+predictor frozen, COCO rows", group="v2-ablation",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "run": "R4",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": 'COCO rows copied for the 20 VOC classes + background',
    "coco_head": True, "frozen": FREEZE_UP_TO_FC7, "frozen_desc": "body + FPN + RPN + fc6",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "linear warmup + cosine, per iteration", "warmup_iters": WARMUP_ITERS,
    "warmup_factor": WARMUP_FACTOR, "eta_min_factor": ETA_MIN_FACTOR,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter on GPU", "hflip_prob": HFLIP_PROB,
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "image_cache": "uint8 memmap", "num_workers": NUM_WORKERS, "pin_memory": True,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED, "gpu": GPU_NAME,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, RUN_DIR, color_jitter=COLOR_JITTER)

log_fit_summary(history)
log_val_and_gap(model, RUN_DIR / "best.pt", val_loader, train_eval_loader)
upload_checkpoint(RUN_DIR / "best.pt", "frcnn-v2-R4")
wandb.finish()

## R5 (optional, only if time allows) — input size: the winner of R2–R4 at 640 px
Measures what 512 px costs in accuracy (especially `mAP_small`) and buys in speed.
**Before running:** set `frozen` in `build_model` (and in the config) to the preset that won R2–R4. It is `FREEZE_UP_TO_ROI` below as a placeholder.

In [ ]:
R5_MIN_SIZE, R5_MAX_SIZE = 640, 1066     # same 800:1333 ratio
R5_FROZEN = FREEZE_UP_TO_ROI             # <- set to the winner of R2-R4

torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(R5_FROZEN, coco_head=True, min_size=R5_MIN_SIZE, max_size=R5_MAX_SIZE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
total_iters = EPOCHS * len(train_loader_aug)
def lr_lambda(step):                                    # multiplier on LR at this iteration
    if step < WARMUP_ITERS:                             # linear warmup: LR * WARMUP_FACTOR -> LR
        return WARMUP_FACTOR + (1 - WARMUP_FACTOR) * step / WARMUP_ITERS
    progress = (step - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)   # 0 -> 1 over the rest of the run
    return ETA_MIN_FACTOR + (1 - ETA_MIN_FACTOR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

RUN_DIR = CKPT_ROOT / "R5_640px"
wandb.init(project=PROJECT, name="v2 R5 | winner at 640 px", group="v2-ablation",
           id=wandb_run_id(RUN_DIR), resume="allow", config={
    "run": "R5",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "COCO rows copied for the 20 VOC classes + background",
    "coco_head": True, "frozen": R5_FROZEN,
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "linear warmup + cosine, per iteration", "warmup_iters": WARMUP_ITERS,
    "warmup_factor": WARMUP_FACTOR, "eta_min_factor": ETA_MIN_FACTOR,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter on GPU", "hflip_prob": HFLIP_PROB,
    "epochs": EPOCHS, "batch": BATCH, "min_size": R5_MIN_SIZE, "max_size": R5_MAX_SIZE, "amp": USE_AMP,
    "image_cache": "uint8 memmap", "num_workers": NUM_WORKERS, "pin_memory": True,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED, "gpu": GPU_NAME,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, RUN_DIR, color_jitter=COLOR_JITTER)

log_fit_summary(history)
log_val_and_gap(model, RUN_DIR / "best.pt", val_loader, train_eval_loader)
upload_checkpoint(RUN_DIR / "best.pt", "frcnn-v2-R5")
wandb.finish()

## Final test evaluation — run ONCE, at the end
Pick the run to report: set its Drive folder, the freezing preset and input size it used. The test metrics are added to that run's W&B summary.

In [ ]:
RUN_DIR = CKPT_ROOT / "R3_up_to_roi_frozen_coco_head"   # <- the chosen run
FROZEN = FREEZE_UP_TO_ROI                               # <- its preset
TEST_MIN_SIZE, TEST_MAX_SIZE = MIN_SIZE, MAX_SIZE       # <- its input size

model, _, _ = build_model(FROZEN, coco_head=True, min_size=TEST_MIN_SIZE, max_size=TEST_MAX_SIZE)
model.load_state_dict(torch.load(RUN_DIR / "best.pt", map_location=DEVICE))
metrics, table = evaluate(model, test_loader, class_metrics=True)

wandb.init(project=PROJECT, id=wandb_run_id(RUN_DIR), resume="must")   # add test results to the same W&B run
wandb.run.summary.update({f"test/{k}": v for k, v in metrics.items()})
wandb.log({"per_class/test": table})
wandb.finish()
print(metrics)